# RAG Demo

Five canonical RAG steps end-to-end with Voyage AI embeddings and ChromaDB.

## Setup (run once before opening this notebook)

```bash
cd ~/Desktop/rag-demo
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env  # then edit .env and put your real Voyage key in
jupyter notebook rag_demo.ipynb
```

Then run cells top-to-bottom.

To simulate the *"some time later"* gap from step 4, restart the kernel after
cell 5 and run only cells 1, 6, 7. Chroma reopens the on-disk store and
retrieval still works.


In [1]:
# Cell 1 — Setup
import os
from pathlib import Path
from dotenv import load_dotenv
import voyageai
import chromadb
from chunker import chunk_by_section

load_dotenv()
if not os.getenv("VOYAGE_API_KEY"):
    raise RuntimeError(
        "VOYAGE_API_KEY missing. Copy .env.example to .env and set the key."
    )

PROJECT_DIR = Path.cwd()
DEMO_FILE = PROJECT_DIR / "demo.md"
CHROMA_DIR = PROJECT_DIR / "chroma_db"
COLLECTION_NAME = "demo"
EMBED_MODEL = "voyage-3-lite"

print("Setup OK. Voyage key loaded; paths configured.")
print(f"  PROJECT_DIR = {PROJECT_DIR}")
print(f"  CHROMA_DIR  = {CHROMA_DIR}")


Setup OK. Voyage key loaded; paths configured.
  PROJECT_DIR = /home/it/Desktop/rag-demo
  CHROMA_DIR  = /home/it/Desktop/rag-demo/chroma_db


In [2]:
# Cell 2 — Load demo.md
text = DEMO_FILE.read_text()
print(f"Loaded {len(text)} chars from {DEMO_FILE.name}\n")
print("--- preview (first 300 chars) ---")
print(text[:300])


Loaded 1366 chars from demo.md

--- preview (first 300 chars) ---
# Photosynthesis

Plants convert sunlight into chemical energy through photosynthesis. Chlorophyll in the leaves absorbs light, and the plant uses that energy to combine carbon dioxide from the air with water from the soil to produce glucose and oxygen. Glucose feeds the plant; oxygen is released in


In [3]:
# Cell 3 — Step 1: chunk by section
chunks = chunk_by_section(text)
print(f"Got {len(chunks)} chunks:\n")
for c in chunks:
    print(f"  [{c['id']}] {c['heading']!r} ({len(c['text'])} chars)")


Got 4 chunks:

  [section-0] 'Photosynthesis' (333 chars)
  [section-1] 'The Roman Aqueducts' (361 chars)
  [section-2] 'Bitcoin Mining' (324 chars)
  [section-3] 'Honeybee Communication' (341 chars)


In [4]:
# Cell 4 — Step 2: embed each chunk via Voyage AI
client = voyageai.Client()  # picks up VOYAGE_API_KEY from env
result = client.embed(
    texts=[c["text"] for c in chunks],
    model=EMBED_MODEL,
    input_type="document",
)
embeddings = result.embeddings
print(f"Embedded {len(embeddings)} chunks.")
print(f"Each embedding has {len(embeddings[0])} dimensions.")
print(f"Total tokens used: {result.total_tokens}")


Embedded 4 chunks.
Each embedding has 512 dimensions.
Total tokens used: 272


In [5]:
# Cell 5 — Step 3: create vector DB and add embeddings
chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Uncomment to wipe and re-index from scratch (e.g., after editing demo.md):
# chroma.delete_collection(COLLECTION_NAME)

collection = chroma.get_or_create_collection(name=COLLECTION_NAME)
collection.add(
    ids=[c["id"] for c in chunks],
    embeddings=embeddings,
    documents=[c["text"] for c in chunks],
    metadatas=[{"heading": c["heading"]} for c in chunks],
)
print(f"Collection {COLLECTION_NAME!r} now contains {collection.count()} items.")
print(f"Persisted to: {CHROMA_DIR}")


Collection 'demo' now contains 4 items.
Persisted to: /home/it/Desktop/rag-demo/chroma_db


In [6]:
# Cell 5b — Persistence-test helper: reopen the existing collection from disk.
#
# After running cells 1–5 once, restart the Jupyter kernel (Kernel → Restart),
# then run cell 1, this cell, and cells 6–7. The `collection.count()` here
# proves the on-disk Chroma store survived the restart — no re-embedding
# needed.
chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma.get_or_create_collection(name=COLLECTION_NAME)
print(f"Reopened collection {COLLECTION_NAME!r} with {collection.count()} items.")


Reopened collection 'demo' with 4 items.


In [7]:
# Cell 6 — Step 4: a user asks a question; embed it
question = "How do plants make food?"

q_emb = client.embed(
    texts=[question],
    model=EMBED_MODEL,
    input_type="query",
).embeddings[0]

print(f"Question: {question}")
print(f"Query embedding has {len(q_emb)} dimensions.")


Question: How do plants make food?
Query embedding has 512 dimensions.


In [8]:
# Cell 7 — Step 5: retrieve the 2 most relevant chunks
results = collection.query(query_embeddings=[q_emb], n_results=2)

print(f"Top 2 results for: {question!r}\n")
for i in range(len(results["ids"][0])):
    chunk_id = results["ids"][0][i]
    distance = results["distances"][0][i]
    heading = results["metadatas"][0][i]["heading"]
    text_preview = results["documents"][0][i][:200].replace("\n", " ")
    print(f"#{i+1} [{chunk_id}] {heading} (distance={distance:.4f})")
    print(f"   {text_preview}...\n")


Top 2 results for: 'How do plants make food?'

#1 [section-0] Photosynthesis (distance=0.8261)
   # Photosynthesis  Plants convert sunlight into chemical energy through photosynthesis. Chlorophyll in the leaves absorbs light, and the plant uses that energy to combine carbon dioxide from the air wi...

#2 [section-3] Honeybee Communication (distance=1.2894)
   # Honeybee Communication  Honeybees use a "waggle dance" to communicate the direction and distance of food sources to other workers. The angle of the dance relative to vertical encodes the direction r...

